In [16]:
import pandas as pd

# Load the incident-level dataset with weather already joined to each incident
df = pd.read_csv('reduced_incident_weather_merged.csv', parse_dates=['date'])

# Define the weather columns we want to keep after aggregation
weather_cols = ['Max Temp (°C)', 'Min Temp (°C)', 'Mean Temp (°C)', 'Total Rain (mm)',
                'Total Snow (cm)', 'Total Precip (mm)', 'Snow on Grnd (cm)',
                'Dir of Max Gust (10s deg)', 'Spd of Max Gust (km/h)']

# Aggregate from incident-level to daily level
# Each row becomes one day with a total incident count and one weather snapshot
# Weather values are the same for all incidents on the same day so we just take the first
daily = df.groupby('date').agg(
    incident_count=('Count', 'sum'),
    day_of_week=('day_of_week', 'first'),
    season=('season', 'first'),
    Year=('Year', 'first'),
    Month=('Month', 'first'),
    Day=('Day', 'first'),
    **{col: (col, 'first') for col in weather_cols}
).reset_index()

print(daily.shape)
daily.head()

(3174, 16)


,date,incident_count,day_of_week,season,Year,Month,Day,Max Temp (°C),Min Temp (°C),Mean Temp (°C),Total Rain (mm),Total Snow (cm),Total Precip (mm),Snow on Grnd (cm),Dir of Max Gust (10s deg),Spd of Max Gust (km/h)
0,2016-12-06,11,Tuesday,Winter,2016,12,6,-18.2,-21.4,-19.8,0.0,0.0,0.0,3,35,48
1,2016-12-07,24,Wednesday,Winter,2016,12,7,-18.6,-25.0,-21.8,0.0,0.0,0.0,3,34,33
2,2016-12-08,27,Thursday,Winter,2016,12,8,-19.8,-26.7,-23.3,0.0,0.0,0.0,3,0,0
3,2016-12-09,41,Friday,Winter,2016,12,9,-20.2,-24.4,-22.3,0.0,0.4,0.4,3,16,30
4,2016-12-10,40,Saturday,Winter,2016,12,10,-18.3,-23.6,-21.0,0.0,0.0,0.0,3,0,0


In [17]:
# Add a binary flag for weekends — 1 if Saturday or Sunday, 0 otherwise
daily['is_weekend'] = daily['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

# Convert day_of_week and season from text to numeric 0/1 columns
# Models can't do math with words like "Tuesday" or "Winter"
# drop_first=True removes one category from each group since it's redundant
daily = pd.get_dummies(daily, columns=['day_of_week', 'season'], prefix=['dow', 'season'], drop_first=True)

print(daily.columns.tolist())

['date', 'incident_count', 'Year', 'Month', 'Day', 'Max Temp (°C)', 'Min Temp (°C)', 'Mean Temp (°C)', 'Total Rain (mm)', 'Total Snow (cm)', 'Total Precip (mm)', 'Snow on Grnd (cm)', 'Dir of Max Gust (10s deg)', 'Spd of Max Gust (km/h)', 'is_weekend', 'dow_Monday', 'dow_Saturday', 'dow_Sunday', 'dow_Thursday', 'dow_Tuesday', 'dow_Wednesday', 'season_Spring', 'season_Summer', 'season_Winter']


In [ ]:
from sklearn.model_selection import train_test_split

# X is all input features — everything the model learns from
# We drop date and Year (not useful patterns) and incident_count (that's what we're predicting)
X = daily.drop(columns=['date', 'Year', 'incident_count'])

# y is the target — the number of incidents we're trying to predict
y = daily['incident_count']

# Split 80% into training and 20% into test
# random_state=42 fixes the random split so results are reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training rows: {X_train.shape[0]}')
print(f'Test rows: {X_test.shape[0]}')

Training rows: 2539
Test rows: 635


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.linear_model import LinearRegression

# Linear Regression is our baseline model
# It finds a straight-line relationship between the input features and incident count
lr = LinearRegression()

# Train the model — it learns the patterns from the training data
lr.fit(X_train, y_train)

# Generate predictions on the test set — data the model has never seen before
y_pred_lr = lr.predict(X_test)

# MAE — average number of incidents we're off by per day (lower is better)
mae  = mean_absolute_error(y_test, y_pred_lr)

# RMSE — similar to MAE but penalizes large errors more heavily (lower is better)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))

# R2 — how much of the variation in incident counts the model explains
# 1.0 is perfect, 0.0 means no better than always guessing the average (higher is better)
r2   = r2_score(y_test, y_pred_lr)

print('=== Linear Regression ===')
print(f'Mean Absolute Error:  {mae:.2f}')
print(f'Root Mean Squared Error: {rmse:.2f}')
print(f'R2 Score:   {r2:.3f}')

=== Linear Regression ===
Mean Absolute Error:  5.49
Root Mean Squared Error: 7.12
R2 Score:   0.360


In [21]:
from sklearn.ensemble import RandomForestRegressor

# Random Forest is our main model — builds hundreds of decision trees and averages their predictions
# This handles complex non-linear patterns that linear regression can't capture
# n_estimators=200 — number of trees to build
# max_depth=10 — limits how deep each tree grows, prevents memorizing the training data
# random_state=42 — reproducibility
# n_jobs=-1 — use all available CPU cores to speed up training
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)

# Train on the same training set as linear regression so the comparison is fair
rf.fit(X_train, y_train)

# Generate predictions on the same test set
y_pred_rf = rf.predict(X_test)

# Same metrics as before — compare these numbers against linear regression
mae  = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2   = r2_score(y_test, y_pred_rf)

print('=== Random Forest ===')
print(f'Mean Absolute Error:  {mae:.2f}')
print(f'Root Mean Squared Error: {rmse:.2f}')
print(f'R2 Score:   {r2:.3f}')

=== Random Forest ===
Mean Absolute Error:  5.24
Root Mean Squared Error: 6.98
R2 Score:   0.385


In [22]:
from sklearn.ensemble import GradientBoostingRegressor

# Gradient Boosting Regressor
# Builds trees sequentially, where each new tree corrects the errors of the previous ones
# More advanced than Random Forest, it focuses on mistakes instead of averaging everything
# n_estimators=200 — number of trees built one after another
# learning_rate=0.1 — how much each tree contributes (lower = slower but more stable learning)
# max_depth=3 — keeps trees small to avoid overfitting
# random_state=42 — makes sure we have the same results every time

gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Train model on training data
gbr.fit(X_train, y_train)
# Generate predictions on test data
y_pred_gbr = gbr.predict(X_test)

# Evaluate performance
mae  = mean_absolute_error(y_test, y_pred_gbr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_gbr))
r2   = r2_score(y_test, y_pred_gbr)

print('=== Gradient Boosting ===')
print(f'Mean Absolute Error:  {mae:.2f}')
print(f'Root Mean Squared Error: {rmse:.2f}')
print(f'R2 Score:   {r2:.3f}')

=== Gradient Boosting ===
Mean Absolute Error:  5.19
Root Mean Squared Error: 6.99
R2 Score:   0.383


In [23]:
from sklearn.linear_model import Ridge

# Ridge Regression
# A version of Linear Regression that adds L2 regularization
# Helps reduce overfitting
# Useful when features are correlated
# alpha=1.0 — strength of regularization (higher = more penalty)

ridge = Ridge(alpha=1.0)

# Train model
ridge.fit(X_train, y_train)
# Predict on test data
y_pred_ridge = ridge.predict(X_test)

# Evaluate performance
mae  = mean_absolute_error(y_test, y_pred_ridge)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
r2   = r2_score(y_test, y_pred_ridge)

print('=== Ridge Regression ===')
print(f'Mean Absolute Error:  {mae:.2f}')
print(f'Root Mean Squared Error: {rmse:.2f}')
print(f'R2 Score:   {r2:.3f}')

=== Ridge Regression ===
Mean Absolute Error:  5.49
Root Mean Squared Error: 7.12
R2 Score:   0.360


In [24]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

# K-Nearest Neighbors Regressor
# Predicts values based on the average of the K most similar data points
# Uses distance between feature values to determine similarity
# Very sensitive to feature scale

# StandardScaler — all contribute equally to distance calculations
# n_neighbors=50 — number of nearby points used for prediction (chosen from early testing)

knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(n_neighbors=50))
])

# Train model
knn_pipeline.fit(X_train, y_train)
# Predict on test data
y_pred_knn = knn_pipeline.predict(X_test)

# Evaluate performance
mae  = mean_absolute_error(y_test, y_pred_knn)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_knn))
r2   = r2_score(y_test, y_pred_knn)

print('=== KNN (k=50, Scaled) ===')
print(f'Mean Absolute Error:  {mae:.2f}')
print(f'Root Mean Squared Error: {rmse:.2f}')
print(f'R2 Score:   {r2:.3f}')

=== KNN (k=50, Scaled) ===
Mean Absolute Error:  5.20
Root Mean Squared Error: 6.97
R2 Score:   0.387


In [25]:
from sklearn.linear_model import SGDRegressor

# Stochastic Gradient Descent regressor - a way to fit linear regressors under convex loss functions
# loss determines what the model is trying to minimize - epsilon_insensitive ignores values within a margin are ignored
# penalty helps to reduce overfitting by changing how weights are adjusted
# random_state adds reproducibility between other regressors
sgd = SGDRegressor(loss="epsilon_insensitive", penalty="l1", random_state=42)

# Trains the model on the same data as the other models, and predicts based on the same test set
sgd.fit(X_train, y_train)
y_pred_sgd = sgd.predict(X_test)

# Provides the same metrics as other models
mae  = mean_absolute_error(y_test, y_pred_sgd)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_sgd))
r2   = r2_score(y_test, y_pred_sgd)

print('=== SGD Regressor ===')
print(f'Mean Absolute Error:  {mae:.2f}')
print(f'Root Mean Squared Error:  {rmse:.2f}')
print(f'R2 Score:  {r2:.3f}')

=== SGD Regressor ===
Mean Absolute Error:  5.62
Root Mean Squared Error:  7.80
R2 Score:  0.232


In [ ]:
from sklearn.ensemble import VotingRegressor

# Voting Regressor - uses different models and gives the average predicted values
# Each estimator is a regressor previously used in the program
vote = VotingRegressor(estimators=[('lr', lr), ('rf', rf), ('sgd', sgd)])

# Trains the model on the same data as the other models, and predicts based on the same test set
vote.fit(X_train, y_train)
y_pred_vote = vote.predict(X_test)

# Provides the same metrics as other models
mae  = mean_absolute_error(y_test, y_pred_vote)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_vote))
r2   = r2_score(y_test, y_pred_vote)

print('=== Voting Regressor ===')
print(f'Mean Absolute Error:  {mae:.2f}')
print(f'Root Mean Squared Error:  {rmse:.2f}')
print(f'R2 Score:  {r2:.3f}')

=== Voting Regressor ===
Mean Absolute Error:  5.16
Root Mean Squared Error:  6.88
R2 Score:  0.403


In [27]:
# Comparison of values from all predictions, together for easier comparisons
print("=== Model Prediction Comparison ===")
# Aligns all numpy arrays into a grid shape
np.set_printoptions(formatter={"float": lambda x: f"{x:6}", "int": lambda x: f"{x:6}"})
print(f'First 10 actual results:         {y_test.values[:10]}')
print(f'Linear Regression predictions:   {y_pred_lr[:10].round(1)}')
print(f'Random Forest predictions:       {y_pred_rf[:10].round(1)}')
print(f'Gradient Boost predictions:      {y_pred_gbr[:10].round(1)}')
print(f'Ridge Regression predictions:    {y_pred_ridge[:10].round(1)}')
print(f'KNN predictions:                 {y_pred_knn[:10].round(1)}')
print(f'Stochastic Gradient predictions: {y_pred_sgd[:10].round(1)}')
print(f'Voting Regressor predictions:    {y_pred_vote[:10].round(1)}')

=== Model Prediction Comparison ===
First 10 actual results:         [    15     14     10     34     15     17     14     23     17     24]
Linear Regression predictions:   [  11.9   19.1    6.8   19.9   19.5    9.7   17.9   16.0   16.5   23.5]
Random Forest predictions:       [  13.1   16.0   10.4   19.3   17.6   12.7   16.8   17.5   17.3   20.5]
Gradient Boost predictions:      [  13.7   17.4    9.5   19.5   16.5   10.9   16.9   18.4   15.8   21.1]
Ridge Regression predictions:    [  11.8   19.1    6.9   19.9   19.5    9.7   17.9   15.9   16.5   23.4]
KNN predictions:                 [  13.6   15.3    9.4   17.7   18.7   12.2   17.0   16.7   15.9   20.7]
Stochastic Gradient predictions: [  17.6   15.8    9.0   20.7   12.7    8.5   13.9   17.2   18.4   22.7]
Voting Regressor predictions:    [  14.2   17.0    8.8   20.0   16.6   10.3   16.2   16.9   17.4   22.2]


In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

#This is just some code to help with the fancy output tables and getting the results for any model
def evaluate_model(model, X_train, X_test, y_train, y_test, name="Model"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"\n{name}")
    print(f'Mean Absolute Error:  {mae:.2f}')
    print(f'Root Mean Squared Error: {rmse:.2f}')
    print(f'R2 Score:   {r2:.3f}')

    return {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

In [29]:
results = []

# 1. Linear Regression
from sklearn.linear_model import LinearRegression
results.append(evaluate_model(
    LinearRegression(),
    X_train, X_test, y_train, y_test,
    "Linear Regression"
))

# 2. Random Forest
from sklearn.ensemble import RandomForestRegressor
results.append(evaluate_model(
    RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    X_train, X_test, y_train, y_test,
    "Random Forest"
))

# 3. Gradient Boosting
from sklearn.ensemble import GradientBoostingRegressor
results.append(evaluate_model(
    GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=3),
    X_train, X_test, y_train, y_test,
    "Gradient Boosting"
))

# 4. Ridge Regression
from sklearn.linear_model import Ridge
results.append(evaluate_model(
    Ridge(alpha=1.0),
    X_train, X_test, y_train, y_test,
    "Ridge Regression"
))

# 5. KNN 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

k_values = [3,20,50]

for k in k_values:
    knn_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsRegressor(n_neighbors=k
        ))
    ])

    results.append(evaluate_model(
        knn_pipeline,
        X_train, X_test, y_train, y_test,
        f"KNN (k={k})"
    ))

# 6. SGD
from sklearn.linear_model import SGDRegressor

sgd = SGDRegressor(
    loss="epsilon_insensitive",
    penalty="l1",
    random_state=42
)

results.append(evaluate_model(
    sgd,
    X_train, X_test, y_train, y_test,
    "SGD Regressor"
))

# 7. Voting
from sklearn.ensemble import VotingRegressor

vote = VotingRegressor(estimators=[
    ('lr', LinearRegression()),
    ('rf', RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)),
    ('sgd', SGDRegressor(loss="epsilon_insensitive", penalty="l1", random_state=42))
])

results.append(evaluate_model(
    vote,
    X_train, X_test, y_train, y_test,
    "Voting Regressor"
))


Linear Regression
Mean Absolute Error:  5.49
Root Mean Squared Error: 7.12
R2 Score:   0.360

Random Forest
Mean Absolute Error:  5.24
Root Mean Squared Error: 6.98
R2 Score:   0.385

Gradient Boosting
Mean Absolute Error:  5.19
Root Mean Squared Error: 6.99
R2 Score:   0.382

Ridge Regression
Mean Absolute Error:  5.49
Root Mean Squared Error: 7.12
R2 Score:   0.360

KNN (k=3)
Mean Absolute Error:  5.93
Root Mean Squared Error: 7.75
R2 Score:   0.242

KNN (k=20)
Mean Absolute Error:  5.24
Root Mean Squared Error: 7.02
R2 Score:   0.378

KNN (k=50)
Mean Absolute Error:  5.20
Root Mean Squared Error: 6.97
R2 Score:   0.387

SGD Regressor
Mean Absolute Error:  5.62
Root Mean Squared Error: 7.80
R2 Score:   0.232

Voting Regressor
Mean Absolute Error:  5.16
Root Mean Squared Error: 6.88
R2 Score:   0.403


In [30]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="R2", ascending=False)

results_df

,Model,MAE,RMSE,R2
8,Voting Regressor,5.160000,6.876531,0.402996
6,KNN (k=50),5.203118,6.970670,0.386539
1,Random Forest,5.240579,6.979398,0.385001
2,Gradient Boosting,5.185341,6.994886,0.382269
5,KNN (k=20),5.236772,7.016303,0.378480
0,Linear Regression,5.488321,7.119654,0.360035
3,Ridge Regression,5.490118,7.121263,0.359746
4,KNN (k=3),5.926509,7.747141,0.242258
7,SGD Regressor,5.618072,7.800917,0.231703
